In [27]:
!pip install -U pyarrow>=21.0.0

In [28]:
!pip install "pydantic<2.12,>=2.0"

In [34]:
!pip install "transformers==4.43.4" "huggingface_hub==0.25.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 74.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.5 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.35.3
    Uninstalling huggingface-hub-0.35.3:
      Successfully uninstalled huggingface-hub-0.35.3
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.0
    Uninstalling transformers-4.57.0:
      Successfully uninstalled transformers-4.57.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the

In [1]:
!pip install --upgrade transformers

  Using cached transformers-4.57.0-py3-none-any.whl.metadata (41 kB)
  Using cached huggingface_hub-0.35.3-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
Using cached transformers-4.57.0-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.35.3-py3-none-any.whl (564 kB)
Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.25.1
    Uninstalling huggingface-hub-0.25.1:
      Successfully uninstalled huggingface-hub-0.25.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.43.4
    Uninstalling transformers-4.43.4:
      Successfully uninstalled transformer

In [1]:
import os
import gc
import re
import json
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras.layers import (
    Input, Dense, Reshape, Flatten, Concatenate,
    Dropout, BatchNormalization, LayerNormalization,
    Add, Layer
    )
from keras.preprocessing import image
from transformers import AutoTokenizer, TFAutoModel
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

# Suppress warnings
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

ModuleNotFoundError: No module named 'tensorflow'

In [3]:
# For reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_seed(42)

In [6]:
class Config:
    N_SPLITS = 5
    
    OUTPUT_DIR = os.getcwd()
    PREPROCESSED_IMAGE_DIR = os.path.join(OUTPUT_DIR, 'preprocessed_images')
    TRAIN_DIR = "/kaggle/input/amazon-ml-challenge-25/train.csv"
    TEST_DIR = "/kaggle/input/amazon-ml-challenge-25/test.csv"
    IMAGES_TRAIN_DIR = "/kaggle/input/amazon-ml-images-50-sample/optimized_50_percent_sample/train"
    IMAGES_TEST_DIR = "/kaggle/input/amazon-ml-images-50-sample/optimized_50_percent_sample/test"
    LOCAL_MODEL_PATH = ""
    
    CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'
    IMG_SIZE = 224
    MAX_TEXT_LEN = 77  # CLIP's max sequence length
    
    BATCH_SIZE = 32
    EPOCHS = 8
    LEARNING_RATE = 1e-4
    
    MAX_FEATURES_TAGS = 500  # Max features for TF-IDF on tags
    MAX_FEATURES_CATS = 200  # Max features for TF-IDF on category candidates

CONFIG = Config()

In [7]:
print("1. Loading data...")
train_df = pd.read_csv(CONFIG.TRAIN_DIR)
test_df = pd.read_csv(CONFIG.TEST_DIR)

1. Loading data...


In [8]:
base_name = lambda file : ".".join(file.split(".")[:-1])
train_samples = [int(base_name(name)) for name in os.listdir(CONFIG.IMAGES_TRAIN_DIR)]
test_samples = [int(base_name(name)) for name in os.listdir(CONFIG.IMAGES_TEST_DIR)]

In [9]:
def sample_data(df, samples) :
    data = []
    for sample in samples :
        data.append(df[df["sample_id"] == sample].values[0])
    return pd.DataFrame(data, columns = df.columns)

train_df = sample_data(train_df, train_samples)
test_df = sample_data(test_df, test_samples)

In [10]:
def extract_features(text: str) -> dict:
    if not isinstance(text, str): text = ""
    features = {}
    PATTERNS = {"item_name": r"Item Name:\s*(.+?)(?:\n|$)", "bullet_points": r"Bullet Point\s*\d*:\s*(.+?)(?=\nBullet Point|$)",}
    for key, pat in PATTERNS.items():
        m = re.findall(pat, text, flags=re.S | re.I)
        if m: features[key] = " ".join([s.strip() for s in m])
    
    # Simple Brand extraction
    brand_match = re.match(r"Item Name:\s*([A-Za-z0-9' -]+)", text)
    if brand_match: features["brand"] = brand_match.group(1).strip()
    else: features["brand"] = "Unknown"

    # Pack count
    pack_match = re.search(r"(?:pack of|set of)\s*(\d+)", text, flags=re.I)
    if pack_match: features["pack_count"] = int(pack_match.group(1))
    else: features["pack_count"] = 1
        
    TAG_PATTERN = r"\b([A-Z][a-z]+(?:[- ][A-Z]?[a-z]+){0,2})\b"
    candidates = re.findall(TAG_PATTERN, text)
    features["tags"] = " ".join(sorted(set([t.strip() for t in candidates if len(t) > 3 and not t.isnumeric()])))
    
    return features

In [11]:
print("2. Applying feature engineering...")
train_features_df = pd.DataFrame([extract_features(text) for text in tqdm(train_df['catalog_content'])])
test_features_df = pd.DataFrame([extract_features(text) for text in tqdm(test_df['catalog_content'])])

2. Applying feature engineering...


  0%|          | 0/37482 [00:00<?, ?it/s]

  0%|          | 0/37499 [00:00<?, ?it/s]

In [12]:
print("3. Vectorizing engineered features...")
# Vectorize 'tags'
tags_vectorizer = TfidfVectorizer(max_features=CONFIG.MAX_FEATURES_TAGS, token_pattern=r'\b[a-zA-Z-]+\b')
train_tags_tfidf = tags_vectorizer.fit_transform(train_features_df['tags'].fillna('')).toarray()
test_tags_tfidf = tags_vectorizer.transform(test_features_df['tags'].fillna('')).toarray()

# Vectorize 'brand' - simple count encoding for this example
brand_counts = train_features_df['brand'].value_counts().to_dict()
train_brand_feat = train_features_df['brand'].map(brand_counts).fillna(1)
test_brand_feat = test_features_df['brand'].map(brand_counts).fillna(1)

# Combine engineered features
engineered_train_feats = np.hstack([
    train_tags_tfidf,
    train_features_df[['pack_count']].fillna(1).values,
    train_brand_feat.values.reshape(-1, 1)
])
engineered_test_feats = np.hstack([
    test_tags_tfidf,
    test_features_df[['pack_count']].fillna(1).values,
    test_brand_feat.values.reshape(-1, 1)
])

# Normalize
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
engineered_train_feats = scaler.fit_transform(engineered_train_feats)
engineered_test_feats = scaler.transform(engineered_test_feats)

print(f"Engineered feature shape: {engineered_train_feats.shape}")


3. Vectorizing engineered features...
Engineered feature shape: (37482, 502)


In [13]:
from huggingface_hub import snapshot_download

local_dir = "clip-model-local"

os.makedirs(local_dir, exist_ok=True)

try:
    snapshot_download(
        repo_id=CONFIG.CLIP_MODEL_NAME,
        local_dir=local_dir,
        local_dir_use_symlinks=False, # Set to False to avoid issues on some systems
        resume_download=True
    )
    CONFIG.LOCAL_MODEL_PATH = os.path.abspath(local_dir)
    print("Download complete!")
    print(f"Model files are saved in: {CONFIG.LOCAL_MODEL_PATH}")

except Exception as e:
    print(f"An error occurred during download: {e}")
    print("Please check your internet connection and firewall settings.")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Download complete!
Model files are saved in: /kaggle/working/clip-model-local


In [ ]:
with tqdm(total = 5, desc = "4. Preparing text and image data...") as pbar :
    # --- Text Tokenization for CLIP ---
    tokenizer = AutoTokenizer.from_pretrained(CONFIG.LOCAL_MODEL_PATH)
    train_text_tokens = tokenizer(text=train_df['catalog_content'].fillna("").tolist(), return_tensors='np', max_length=CONFIG.MAX_TEXT_LEN, padding='max_length', truncation=True)
    pbar.update(1)
    test_text_tokens = tokenizer(text=test_df['catalog_content'].fillna("").tolist(), return_tensors='np', max_length=CONFIG.MAX_TEXT_LEN, padding='max_length', truncation=True)
    pbar.update(1)

    # --- Image Preprocessing & Caching ---
    os.makedirs(CONFIG.PREPROCESSED_IMAGE_DIR, exist_ok=True)
    def preprocess_image(sample_id, image_dir):
        img_path = os.path.join(image_dir, f"{sample_id}.jpg")
        filepath = os.path.join(CONFIG.PREPROCESSED_IMAGE_DIR, f"{sample_id}.npy")
        try:
            if os.path.exists(filepath):
                return np.load(filepath)

            img = image.load_img(img_path, target_size=(CONFIG.IMG_SIZE, CONFIG.IMG_SIZE))
            img_array = image.img_to_array(img)
            img_array = np.array(img, dtype=np.float32) / 255.0
            return img_array
        except (FileNotFoundError, OSError, ValueError) as e :
            print(f"[WARN] Could not process {sample_id}: {e}")
            return np.zeros((CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3), dtype=np.float32)

    def preprocess_images(image_dir, df) :
        array = []
        count = 0
        for i in tqdm(df["sample_id"].tolist(), total = df.shape[0], desc = "Image Processing") :
            array.append(preprocess_image(i, image_dir))
            count += 1
        return np.array(array)

    train_images = preprocess_images(CONFIG.IMAGES_TRAIN_DIR, train_df)
    pbar.update(1)

    test_images = preprocess_images(CONFIG.IMAGES_TEST_DIR, test_df)
    pbar.update(1)

    y = train_df['price'].values
    y_log = np.log1p(y) # Use log(1+price) for stability
    pbar.update(1)

4. Preparing text and image data...:   0%|          | 0/5 [00:00<?, ?it/s]

Image Processing:   0%|          | 0/37482 [00:00<?, ?it/s]

[WARN] Could not process 31065: 150528 requested and 39904 written
[WARN] Could not process 292598: [Errno 28] No space left on device
[WARN] Could not process 264497: [Errno 28] No space left on device
[WARN] Could not process 70410: [Errno 28] No space left on device
[WARN] Could not process 201100: [Errno 28] No space left on device
[WARN] Could not process 39017: [Errno 28] No space left on device
[WARN] Could not process 34428: [Errno 28] No space left on device
[WARN] Could not process 235340: [Errno 28] No space left on device
[WARN] Could not process 198867: [Errno 28] No space left on device
[WARN] Could not process 219121: [Errno 28] No space left on device
[WARN] Could not process 63016: [Errno 28] No space left on device
[WARN] Could not process 142952: [Errno 28] No space left on device
[WARN] Could not process 38450: [Errno 28] No space left on device
[WARN] Could not process 230455: [Errno 28] No space left on device
[WARN] Could not process 194820: [Errno 28] No space l

In [ ]:
class MultiHeadCrossAttention(Layer):
    def __init__(self, d_model=512, num_heads=8, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_dense = Dense(d_model)
        self.key_dense = Dense(d_model)
        self.value_dense = Dense(d_model)
        self.combine_heads = Dense(d_model)
        self.layernorm = LayerNormalization()
        self.add = Add()

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.head_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, query_input, key_input, value_input):
        batch_size = tf.shape(query_input)[0]
        query = self.query_dense(query_input)
        key = self.key_dense(key_input)
        value = self.value_dense(value_input)
        
        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)
        
        attention_output = self.attention(query, key, value)
        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention_output, (batch_size, -1, self.d_model))
        
        combined = self.combine_heads(concat_attention)
        # Add & Norm
        output = self.layernorm(self.add([query_input, combined]))
        return output

In [ ]:
def create_model():
    # --- Define Inputs ---
    image_input = Input(shape=(CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3), name="image_input")
    text_ids_input = Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name="text_ids_input")
    text_mask_input = Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name="text_mask_input")
    engineered_input = Input(shape=(engineered_train_feats.shape[1],), name="engineered_input")
    
    # --- Load Pre-trained CLIP Model ---
    clip_model = TFAutoModel.from_pretrained(CONFIG.CLIP_MODEL_NAME)
    
    # Freeze CLIP     clip_model.text_model.trainable = False
    clip_model.vision_model.trainable = False
    
    # --- Get Embeddings ---
    text_embeds = clip_model.text_model(input_ids=text_ids_input, attention_mask=text_mask_input).pooler_output
    image_embeds = clip_model.vision_model(pixel_values=image_input).pooler_output
    
    # Add a sequence dimension for attention
    text_embeds_seq = Reshape((1, -1))(text_embeds)
    image_embeds_seq = Reshape((1, -1))(image_embeds)

    # --- Multi-Head Cross-Modal Attention Block ---
    attention_layer = MultiHeadCrossAttention(d_model=text_embeds.shape[-1], num_heads=8)
    text_to_image_features = attention_layer(text_embeds_seq, image_embeds_seq, image_embeds_seq)
    image_to_text_features = attention_layer(image_embeds_seq, text_embeds_seq, text_embeds_seq)
    
    fused_features = Concatenate()([
        Flatten()(text_to_image_features),
        Flatten()(image_to_text_features)
    ])
    
    # --- Final Fusion and Regression Head ---
    all_features = Concatenate()([fused_features, engineered_input])
    
    x = BatchNormalization()(all_features)
    x = Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-5))(x)
    x = Dropout(0.4)(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-5))(x)
    
    output = Dense(1, activation='relu', name='price_output')(x) # ReLU to ensure positive price
    
    model = keras.Model(
        inputs=[image_input, text_ids_input, text_mask_input, engineered_input],
        outputs=output
    )
    
    return model

In [ ]:
# Custom SMAPE Metric for monitoring
def smape_metric(y_true, y_pred):
    y_true = tf.expm1(y_true) # Reverse log transform
    y_pred = tf.expm1(y_pred)
    numerator = tf.abs(y_pred - y_true)
    denominator = (tf.abs(y_true) + tf.abs(y_pred)) / 2.0
    return tf.reduce_mean(numerator / (denominator + 1e-8)) * 100.0

In [ ]:
print("5. Starting training with train test split...")
kf = KFold(n_splits=CONFIG.N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(tqdm(kf.split(train_df), desc = "SKF")):
    print(f"\n===== FOLD {fold+1}/{CONFIG.N_SPLITS} =====")
    
    # --- Prepare Data for Fold ---
    X_train = {
        'image_input': train_images[train_idx],
        'text_ids_input': train_text_tokens['input_ids'][train_idx],
        'text_mask_input': train_text_tokens['attention_mask'][train_idx],
        'engineered_input': engineered_train_feats[train_idx]
    }
    y_train_fold = y_log[train_idx]
    
    X_val = {
        'image_input': train_images[val_idx],
        'text_ids_input': train_text_tokens['input_ids'][val_idx],
        'text_mask_input': train_text_tokens['attention_mask'][val_idx],
        'engineered_input': engineered_train_feats[val_idx]
    }
    y_val_fold = y_log[val_idx]
    
    # --- Build and Compile Model ---
    keras.backend.clear_session()
    model = create_model()
    optimizer = keras.optimizers.AdamW(learning_rate=CONFIG.LEARNING_RATE)
    # Huber loss is a good proxy for MAE, robust to outliers
    model.compile(optimizer=optimizer, loss='huber', metrics=[smape_metric])
    
    # --- Callbacks ---
    lr_reducer = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
    early_stopper = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)
    
    # --- Train Model ---
    model.fit(
        X_train, y_train_fold,
        validation_data=(X_val, y_val_fold),
        epochs=CONFIG.EPOCHS,
        batch_size=CONFIG.BATCH_SIZE,
        callbacks=[lr_reducer, early_stopper]
    )
    
    # --- Predict and Store ---
    oof_preds[val_idx] = model.predict(X_val).flatten()
    
    X_test = {
        'image_input': test_images,
        'text_ids_input': test_text_tokens['input_ids'],
        'text_mask_input': test_text_tokens['attention_mask'],
        'engineered_input': engineered_test_feats
    }
    test_preds += model.predict(X_test).flatten() / CONFIG.N_SPLITS
    
    # --- Clean up ---
    del model
    gc.collect()
print("\nTraining complete.")

In [ ]:
print("5. Starting training with K-Fold Cross-Validation...")
kf = KFold(n_splits=CONFIG.N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(tqdm(kf.split(train_df), desc = "SKF")):
    print(f"\n===== FOLD {fold+1}/{CONFIG.N_SPLITS} =====")
    
    # --- Prepare Data for Fold ---
    X_train = {
        'image_input': train_images[train_idx],
        'text_ids_input': train_text_tokens['input_ids'][train_idx],
        'text_mask_input': train_text_tokens['attention_mask'][train_idx],
        'engineered_input': engineered_train_feats[train_idx]
    }
    y_train_fold = y_log[train_idx]
    
    X_val = {
        'image_input': train_images[val_idx],
        'text_ids_input': train_text_tokens['input_ids'][val_idx],
        'text_mask_input': train_text_tokens['attention_mask'][val_idx],
        'engineered_input': engineered_train_feats[val_idx]
    }
    y_val_fold = y_log[val_idx]
    
    # --- Build and Compile Model ---
    keras.backend.clear_session()
    model = create_model()
    optimizer = keras.optimizers.AdamW(learning_rate=CONFIG.LEARNING_RATE)
    # Huber loss is a good proxy for MAE, robust to outliers
    model.compile(optimizer=optimizer, loss='huber', metrics=[smape_metric])
    
    # --- Callbacks ---
    lr_reducer = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
    early_stopper = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)
    
    # --- Train Model ---
    model.fit(
        X_train, y_train_fold,
        validation_data=(X_val, y_val_fold),
        epochs=CONFIG.EPOCHS,
        batch_size=CONFIG.BATCH_SIZE,
        callbacks=[lr_reducer, early_stopper]
    )
    
    # --- Predict and Store ---
    oof_preds[val_idx] = model.predict(X_val).flatten()
    
    X_test = {
        'image_input': test_images,
        'text_ids_input': test_text_tokens['input_ids'],
        'text_mask_input': test_text_tokens['attention_mask'],
        'engineered_input': engineered_test_feats
    }
    test_preds += model.predict(X_test).flatten() / CONFIG.N_SPLITS
    
    # --- Clean up ---
    del model
    gc.collect()
print("\nTraining complete.")

In [ ]:
# Reverse log transform for final predictions
oof_preds_final = np.expm1(oof_preds)
test_preds_final = np.expm1(test_preds)

# Ensure no negative prices
test_preds_final[test_preds_final < 0] = 0 

# Calculate overall OOF SMAPE
final_oof_smape = smape_metric(y, oof_preds_final).numpy()
print(f"Overall Out-of-Fold SMAPE: {final_oof_smape:.4f}")

# --- Create Submission File ---
submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': test_preds_final
})
submission_df.to_csv('submission.csv', index=False)
print("\nSubmission file 'submission.csv' created successfully.")
print("Top 5 predictions:")
print(submission_df.head())